In [8]:
import sys
import os
import time
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(os.path.join('..')))
import scripts.simulator as simulator
importlib.reload(simulator)
from scripts.simulator import simulate, simulate_fast, simulate_batch

In [6]:
M = 250
R = 40
TOTAL = M * R
PARAMS = dict(beta=0.3, gamma=0.15, rho=0.7)

# ── Warm up Numba (compilation cost, not counted in benchmark) ────────────────
print("Warming up Numba JIT (one-time cost)...")
simulate_fast(**PARAMS)
print("Done.\n")

# ── Benchmark 1: old sequential ───────────────────────────────────────────────
rng = np.random.default_rng(42)
t0 = time.perf_counter()
old_results = [simulate(**PARAMS, rng=rng) for _ in range(TOTAL)]
old_time = time.perf_counter() - t0
print(f"Old (sequential, pure Python): {old_time:.2f}s  ({old_time/TOTAL*1000:.1f} ms/run)")

# ── Benchmark 2: new sequential (Numba only, no parallelism) ─────────────────
rng = np.random.default_rng(42)
t0 = time.perf_counter()
new_seq_results = [simulate_fast(**PARAMS, rng=rng) for _ in range(TOTAL)]
new_seq_time = time.perf_counter() - t0
print(f"New (sequential, Numba JIT):   {new_seq_time:.2f}s  ({new_seq_time/TOTAL*1000:.1f} ms/run)")

# ── Benchmark 3: new parallel (Numba + joblib, M draws × R replicates) ────────
rng = np.random.default_rng(42)
thetas = np.full((M, 3), [PARAMS["beta"], PARAMS["gamma"], PARAMS["rho"]])
t0 = time.perf_counter()
new_par_results = simulate_batch(thetas, n_replicates=R, rng=rng)
new_par_time = time.perf_counter() - t0
print(f"New (parallel,  Numba+joblib): {new_par_time:.2f}s  ({new_par_time/TOTAL*1000:.1f} ms/run)")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\nTotal simulations per benchmark: {TOTAL:,}")
print(f"Speedup (Numba only):            {old_time/new_seq_time:.1f}×")
print(f"Speedup (Numba + parallel):      {old_time/new_par_time:.1f}×")

Warming up Numba JIT (one-time cost)...
Done.

Old (sequential, pure Python): 241.50s  (24.2 ms/run)
New (sequential, Numba JIT):   7.31s  (0.7 ms/run)
New (parallel,  Numba+joblib): 0.62s  (0.1 ms/run)

Total simulations per benchmark: 10,000
Speedup (Numba only):            33.0×
Speedup (Numba + parallel):      387.5×
